# SatQuery: linear versus MLP
Uses existing features and labels; all code and results are saved in the EDU-backed SatQuery folder. See docs/nonlinear-comparison.md for selection and resume rules.

In [ ]:
from pathlib import Path
import gc, os, subprocess, sys, urllib.request, zipfile
PIPELINE=Path('/content/drive/MyDrive/SatQuery (1)/pipeline-5000')
if not PIPELINE.is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
assert PIPELINE.is_dir(), 'Connect the Drive with the EDU SatQuery shortcut first'
# Resolve a revision once, then retain its full source beside the experiment data.
REPO='https://github.com/raviasha/Sat_Query.git'
REVISION=subprocess.check_output(['git','ls-remote',REPO,'refs/heads/codex/nonlinear-coverage-head'],text=True).split()[0]
CODE=PIPELINE.parent/'code'
CODE.mkdir(exist_ok=True)
archive=CODE/f'satquery-{REVISION}.zip'
if not archive.exists():
    temporary=archive.with_suffix('.partial')
    urllib.request.urlretrieve(f'https://codeload.github.com/raviasha/Sat_Query/zip/{REVISION}',temporary)
    temporary.replace(archive)
RUNTIME=Path('/content/satquery-comparison-source')
RUNTIME.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z: z.extractall(RUNTIME)
SOURCE=RUNTIME/f'Sat_Query-{REVISION}'
assert (SOURCE/'src/satquery/compare_heads.py').is_file()
print('SOURCE_REVISION',REVISION,'SAVED_CODE',archive,flush=True)
# Release previous notebook tensors before the standalone runner loads verified splits.
for name in ('splits','test','z','pred','baseline','saved','model','restored'):
    globals().pop(name,None)
gc.collect()
import torch
torch.cuda.empty_cache()
assert torch.cuda.is_available(), 'Connect a GPU for this comparison'
env=os.environ.copy(); env['PYTHONPATH']=str(SOURCE/'src')
subprocess.run([sys.executable,'-u','-m','satquery.compare_heads','--pipeline',str(PIPELINE),
                '--device','cuda','--seeds','17','29','43','--source-revision',REVISION],
               env=env,check=True)
